In [1]:
# Assuming you have dataset.py and model.py in your current directory
from data import FIFASequenceDataset
from data.dataGenerator import FIFAWC22

data_dir = "FIFA World Cup 2022/Processed Tensors"
match_files = [
    "10510.pt"
] 

# 2. Instantiate the Dataset
print("Initializing Dataset...")
dataset = FIFASequenceDataset(
    data_dir=data_dir, 
    match_files=match_files, 
    target_frames=100
)

Initializing Dataset...
Loading matches into memory.
Loaded 28 total plays across 1 matches!


In [2]:
len(dataset)

28

In [3]:
print("\nSample Data Row (Decoded from Tensor):")
sample_item = dataset[0]  # Grab the first full play

features_tensor = sample_item['features']      # Shape: [100, 23, 9]
label = sample_item['supcon_label']
seq_id = sample_item['sequence_id']
# 1. Select a specific frame and player to inspect
frame_idx = 0   # The first frame of the sequence
player_idx = 2  # 0 is the ball, 1-11 are Home, 12-22 are Away

# Extract the 9 features for this specific player at this specific frame
player_features = features_tensor[frame_idx, player_idx]

# 2. Map the 9 tensor channels back to their semantic names
# The first 7 come from your dataGenerator, the last 2 were added in dataset.py
feature_names = [
    "x", 
    "y", 
    "z", 
    "speed", 
    "visibility", 
    "is_attacking", 
    "dist_to_goal", 
    "is_home", 
    "is_away"
]

# 3. Print the validation row
# .item() converts 0-d PyTorch tensors back to standard Python numbers
print(f"  seq_id: {seq_id.item() if hasattr(seq_id, 'item') else seq_id}")
print(f"  frame_idx: {frame_idx}")
print(f"  player_id: {player_idx}")

for name, value in zip(feature_names, player_features):
    print(f"  {name}: {value.item()}")

print(f"  supcon_label: {label}")


Sample Data Row (Decoded from Tensor):
  seq_id: 10
  frame_idx: 0
  player_id: 2
  x: 0.6410857439041138
  y: 0.29208824038505554
  z: 0.0
  speed: 0.14100000262260437
  visibility: 1.0
  is_attacking: 1.0
  dist_to_goal: 0.4627472460269928
  is_home: 1.0
  is_away: 0.0
  supcon_label: CR_I_D


In [4]:
import torch
from torch.utils.data import DataLoader

# 1. Create the DataLoader
batch_size = 16  # Adjust based on your GPU VRAM (16 or 32 is a good start)
dataloader = DataLoader(
    dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=2,   # Set to 0 if you get multi-processing errors on Windows
    pin_memory=True  # Speeds up CPU to GPU data transfer
)

print(f"Created DataLoader with {len(dataloader)} batches (Batch Size: {batch_size}).")

Created DataLoader with 2 batches (Batch Size: 16).


In [5]:
from Hierarchical_Play_Encoder import HierarchicalPlayEncoder

# 2. Setup Device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Initialize the Model
model = HierarchicalPlayEncoder(
    d_model=128, 
    n_heads=4, 
    frame_layers=2, 
    play_layers=2
).to(device)

print("Model initialized and moved to device successfully!")

Using device: cuda


/home/mostafa/miniconda3/envs/pytorch_env/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Model initialized and moved to device successfully!


In [7]:
# 4. Fetch a single batch from the dataloader
batch = next(iter(dataloader))

# Extract features and send ONLY numerical tensors to the GPU
# Shape expected: [Batch, 100, 23, 9]
features = batch['features'].to(device)

# Labels and sequence IDs are lists of strings, keep them on the CPU
labels = batch['supcon_label']
sequence_ids = batch['sequence_id']

print(f"Input features shape: {features.shape}")
print(f"Input labels length: {len(labels)} (Sample: {labels[:3]})")
print("-" * 40)

# 5. Run the forward pass
final_embedding, projected_embedding = model(features)

# 6. Validate the Output Shapes
print(f"Final Embedding Shape: {final_embedding.shape} (Expected: [{features.shape[0]}, 128])")
print(f"Projected Embedding Shape: {projected_embedding.shape} (Expected: [{features.shape[0]}, 64])")

if final_embedding.shape[1] == 128 and projected_embedding.shape[1] == 64:
    print("\n✅ Forward pass successful! Tensor shapes are perfectly aligned.")
else:
    print("\n❌ Warning: Output shapes do not match expectations.")

Input features shape: torch.Size([16, 100, 23, 9])
Input labels length: 16 (Sample: ['SH_S_B __ SH_S_S', 'CR_O_D', 'CR_I_B __ SH_F_S'])
----------------------------------------
Final Embedding Shape: torch.Size([16, 128]) (Expected: [16, 128])
Projected Embedding Shape: torch.Size([16, 64]) (Expected: [16, 64])

✅ Forward pass successful! Tensor shapes are perfectly aligned.
